<a href="https://colab.research.google.com/github/viniciuspaulo99/desafio1_api_python/blob/main/Desafio1_API_Python.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# Instala as bibliotecas usadas no guia.
!pip install -q gspread faker

In [3]:
# Importa as ferramentas para autenticação, geração de dados e sorteios.
import json, gspread
from google.colab import userdata
from faker import Faker
import random

# Lê a credencial salva nos Secrets e conecta à planilha.
info = json.loads(userdata.get("GCP_SA_JSON"))
gc = gspread.service_account_from_dict(info)
# Exibe o e-mail da conta autenticada.
print("Conectado como:", info["client_email"])

Conectado como: desafio1-python@peaceful-storm-513817-k4.iam.gserviceaccount.com


In [4]:
# Informe o ID da planilha e o nome que será dado à aba.
SHEET_ID = "1qP4TsGSLZbt3RCRxSXaZCM39DmmP0gFihjj6ffP685Q"
NOME_TABELA = "estacoes"   # nome da tabela na ficha do tema, ex.: "vagas"

# Abre a planilha e seleciona sua primeira aba.
sh = gc.open_by_key(SHEET_ID)
ws = sh.sheet1
# Renomeia a aba e exibe os dados da planilha.
ws.update_title(NOME_TABELA)     # a aba passa a ter o nome da tabela
print(sh.title, "|", ws.title, "|", sh.url)

Desafio_01_Meteorologia | estacoes | https://docs.google.com/spreadsheets/d/1qP4TsGSLZbt3RCRxSXaZCM39DmmP0gFihjj6ffP685Q


In [5]:
# Substitua pelos nomes EXATOS de coluna da ficha do seu tema
# Define as colunas que formarão o cabeçalho da tabela.
COLUNAS = ["estacao_id", "nome_estacao", "cidade", "pais", "latitude", "longitude", "altitude_m"]

# Escreve o cabeçalho na primeira célula e confere o resultado.
ws.update(range_name="A1", values=[COLUNAS])
print("Cabeçalho confere:", ws.row_values(1) == COLUNAS)

Cabeçalho confere: True


In [11]:
# Configura os geradores aleatórios para produzir resultados reproduzíveis.
fake = Faker("pt_BR")
Faker.seed(42)
random.seed(42)

def gerar_linha(i):
    # Cria um registro fictício para as colunas definidas no esquema.
    # Personalize de acordo com o seu esquema de dados

    # Lista com os dados exatos que você deseja inserir
    dados_exatos = [
        {"cidade": "Braga", "pais": "Portugal", "latitude": 37.8338, "longitude": -8.4127, "altitude_m": 590},
        {"cidade": "Córdoba", "pais": "Argentina", "latitude": -38.677, "longitude": -59.1101, "altitude_m": 2109},
        {"cidade": "Marselha", "pais": "França", "latitude": 44.1635, "longitude": 4.6858, "altitude_m": 105},
        {"cidade": "Paris", "pais": "França", "latitude": 49.6476, "longitude": 4.7372, "altitude_m": 3406},
        {"cidade": "Mendoza", "pais": "Argentina", "latitude": -27.3641, "longitude": -58.585, "altitude_m": 254},
        {"cidade": "Toronto", "pais": "Canadá", "latitude": 51.9661, "longitude": -97.2022, "altitude_m": 971},
        {"cidade": "Sales", "pais": "Brasil", "latitude": -32.0681, "longitude": -54.5841, "altitude_m": 3128},
        {"cidade": "Frankfurt", "pais": "Alemanha", "latitude": 54.2149, "longitude": 7.7564, "altitude_m": 1154},
        {"cidade": "Mumbai", "pais": "Índia", "latitude": 30.7811, "longitude": 75.5793, "altitude_m": 544},
        {"cidade": "Lisboa", "pais": "Portugal", "latitude": 38.4379, "longitude": -8.9435, "altitude_m": 1491}
    ]

    registro = dados_exatos[i - 1]

    return {
        "estacao_id": i,
        "nome_estacao": f"EST-{i:05d}",
        "cidade": registro["cidade"],
        "pais": registro["pais"],
        "latitude": registro["latitude"],
        "longitude": registro["longitude"],
        "altitude_m": registro["altitude_m"]
    }

# Gera 10 registros numerados de 1 a 10.
linhas = [gerar_linha(i) for i in range(1, 11)]

# Organiza os valores na ordem do cabeçalho; nome de coluna incorreto gera KeyError.
valores = [[linha[c] for c in COLUNAS] for linha in linhas]
# Acrescenta os registros à planilha e informa quantos foram enviados.
ws.append_rows(valores, value_input_option="RAW")
print(len(valores), "linhas enviadas")

10 linhas enviadas


In [10]:
# Lê todos os valores da aba para conferir o conteúdo enviado.
valores_lidos = ws.get_all_values()
# Exibe a quantidade de linhas, confere o cabeçalho e mostra exemplos.
print("Linhas (cabeçalho + dados):", len(valores_lidos))    # esperado: 11
print("Cabeçalho confere:", valores_lidos[0] == COLUNAS)
print(valores_lidos[:3])

Linhas (cabeçalho + dados): 21
Cabeçalho confere: True
[['estacao_id', 'nome_estacao', 'cidade', 'pais', 'latitude', 'longitude', 'altitude_m'], ['1', 'EST-00001', 'Alves', 'Brasil', '-32,0496', '-62,5489', '914'], ['2', 'EST-00002', 'New York', 'Estados Unidos', '41,2408', '-73,1458', '356']]
